# 08 — Diagnostik: kenapa cabang teks tidak terpakai?

**Belum pernah dijalankan.** Hitungan menit, tanpa training. Jalankan setelah notebook
07.

## Pertanyaannya

Notebook 07 menetapkan bahwa cabang teks di dalam fusion hanya memakai 9-12% potensi
yang tersedia, bahkan setelah 200 epoch penuh tanpa early stopping. Yang belum diketahui
adalah **di mana** informasinya hilang. Ada dua kemungkinan yang butuh perbaikan berbeda:

**Hipotesis A — cabang teksnya kolaps.** `text.features(x)` menghasilkan vektor 128 yang
nyaris sama untuk semua dokumen. Kalau begitu, menolkannya memang tidak mengubah apa
pun, dan menghilangkan citra tidak menyisakan apa pun. Perbaikannya: inisialisasi cabang
teks dari checkpoint CNN1D yang sudah terlatih.

**Hipotesis B — cabang teksnya baik, head-nya yang mengabaikan.** Fiturnya masih membawa
informasi kelas, tapi MLP fusion belajar memberinya bobot mendekati nol. Perbaikannya
berbeda sama sekali: normalisasi per cabang sebelum digabung, atau gerbang eksplisit.

**Hipotesis C — skalanya tidak seimbang.** Kalau fitur citra bernorma jauh lebih besar
daripada fitur teks, head concat secara alami akan didominasi citra tanpa pernah
"memutuskan" apa pun. Ini bisa terjadi bersama A atau B.

## Cara membedakannya

Empat pengukuran pada fitur 128 dimensi, dengan **CNN1D standalone sebagai acuan** —
model itu mencapai OA 0,7345 sendirian, jadi fiturnya kita tahu berfungsi.

1. **Varians per dimensi.** Kolaps → mendekati nol.
2. **Unit mati.** Keluaran ReLU, jadi dimensi yang selalu nol itu mungkin.
3. **Norma dan rank efektif.** Menguji hipotesis C sekaligus mengukur berapa dimensi
   yang benar-benar dipakai.
4. **Linear probe.** Ini yang menentukan: latih classifier linear di atas fitur yang
   **dibekukan**, fit pada split train, evaluasi pada test. Kalau probe mencapai akurasi
   tinggi, fiturnya baik dan head-nya yang salah (hipotesis B). Kalau probe gagal,
   cabangnya yang kolaps (hipotesis A).

## Tidak ada yang tertimpa

Notebook ini **hanya membaca** checkpoint. Satu-satunya yang ditulis:
`reports/diagnostik_cabang_teks.csv` dan `reports/figures/09_*.png`. Tidak ada
checkpoint, log, atau CSV lama yang disentuh.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src import datasets as DS
from src import splits as S
from src import train as T
from src.config import CONFIG, set_seed
from src.models import CNN1D, FusionModel, build_transform
from src.utils import save_fig

SEED = 42
set_seed(SEED)

manifest = pd.read_csv(CONFIG["data_processed"] / "manifest.csv")
labels = manifest["kelas"].tolist()
y = DS.label_indices(labels)
split = S.load_split(SEED, labels)
device = T.pick_device()
transform = build_transform()
seq_path = CONFIG["data_processed"] / "sequences_500x300_fp16.npy"
paths = manifest["path_citra"].tolist()
print(f"seed {SEED} | train {len(split['train'])} | test {len(split['test'])} | {device}")

In [ ]:
# Hanya membaca. Tidak ada satu pun file lama yang ditulis notebook ini.
perlu = ["CNN1D_seed42.pt", "FUSION-concat_seed42.pt", "FUSION-sum_seed42.pt",
         "FUSION-concat-noES_seed42.pt"]
hilang = [f for f in perlu if not (CONFIG["models"] / f).exists()]
assert not hilang, f"checkpoint tidak ada: {hilang}. Jalankan notebook 05 dan 07 dulu."
print("checkpoint yang dibaca (tidak ditulis):")
for f in perlu:
    print("  ", f)
print("\nyang akan ditulis: reports/diagnostik_cabang_teks.csv, figures/09_*.png")

In [ ]:
def muat_fusion(strategy, nama):
    m = FusionModel(strategy=strategy, pretrained_image=False)
    ck = torch.load(CONFIG["models"] / nama, map_location="cpu", weights_only=True)
    m.load_state_dict(ck["state_dict"])
    return m.to(device).eval(), ck

m_std = CNN1D()
ck = torch.load(CONFIG["models"] / f"CNN1D_seed{SEED}.pt", map_location="cpu",
                weights_only=True)
m_std.load_state_dict(ck["state_dict"])
m_std = m_std.to(device).eval()
print(f"CNN1D standalone      : epoch {ck['epoch']}, val_oa {ck['val_oa']:.4f}")

m_concat, ck1 = muat_fusion("concat", f"FUSION-concat_seed{SEED}.pt")
m_noES, ck2 = muat_fusion("concat", f"FUSION-concat-noES_seed{SEED}.pt")
m_sum, ck3 = muat_fusion("sum", f"FUSION-sum_seed{SEED}.pt")
for nama, c in [("concat run pertama", ck1), ("concat-noES", ck2), ("sum", ck3)]:
    print(f"{nama:22s}: epoch {c['epoch']}, val_oa {c['val_oa']:.4f}")

## Ekstraksi fitur

Untuk tiap model, kumpulkan vektor 128 dimensi cabang teks di seluruh split. Untuk model
fusion juga cabang citranya (setelah proyeksi 1280 → 128), supaya skala kedua cabang bisa
dibandingkan langsung — itu yang menguji hipotesis C.

In [ ]:
def loader(idx, batch=CONFIG["batch_size"]):
    img = DS.ImageDataset(paths, y, idx, transform)
    txt = DS.SequenceDataset(seq_path, y, idx)
    return torch.utils.data.DataLoader(DS.PairDataset(img, txt), batch_size=batch,
                                       num_workers=CONFIG["num_workers"])

@torch.no_grad()
def ambil_fitur(model, idx, jenis="teks"):
    """jenis: 'teks' | 'citra' | 'standalone'. Mengembalikan (N, 128) float32."""
    out, lab = [], []
    for (img, txt), yb in loader(idx):
        img, txt = img.to(device), txt.to(device)
        if jenis == "standalone":
            f = model.features(txt)
        elif jenis == "teks":
            f = model.text.features(txt)
        else:
            f = model.project(model.image.features(img))
        out.append(f.float().cpu().numpy())
        lab.append(yb.numpy())
    return np.concatenate(out), np.concatenate(lab)

F = {}
F["CNN1D standalone"] = ambil_fitur(m_std, split["test"], "standalone")
F["concat: cabang teks"] = ambil_fitur(m_concat, split["test"], "teks")
F["concat: cabang citra"] = ambil_fitur(m_concat, split["test"], "citra")
F["noES: cabang teks"] = ambil_fitur(m_noES, split["test"], "teks")
F["noES: cabang citra"] = ambil_fitur(m_noES, split["test"], "citra")
F["sum: cabang teks"] = ambil_fitur(m_sum, split["test"], "teks")
F["sum: cabang citra"] = ambil_fitur(m_sum, split["test"], "citra")

for k, (f, _) in F.items():
    print(f"{k:24s} {f.shape}")

## 1-3. Varians, unit mati, norma, rank efektif

**Rank efektif** dihitung sebagai participation ratio dari nilai eigen kovarians:
`(Σλ)² / Σλ²`. Nilainya 128 berarti varians tersebar rata ke semua dimensi; nilai 1
berarti seluruh varians ada di satu arah.

> **Tidak ada satu metrik pun yang cukup sendiri.** Ini diverifikasi pada fitur sintetis
> yang kolapsnya sudah diketahui, dan hasilnya:
>
> | Bentuk kolaps | `std_rata2` | `cos_antar_dokumen` | `rank_efektif` |
> |---|---|---|---|
> | Isotropik (varians kecil di semua arah) | **0,001 — turun 580x** | **1,000** | 105,6 — *buta* |
> | Terarah (varians di satu arah) | 0,457 — *buta* | 0,494 — *buta* | **1,56** |
> | Sehat, sebagai acuan | 0,582 | 0,321 | 104,8 |
>
> Rank efektif bersifat **scale-invariant**, jadi ia tidak bisa melihat varians yang
> mengecil merata. Sebaliknya `std` dan `cos` tidak bisa melihat varians yang
> terkonsentrasi di satu arah. Ketiga kolom harus dibaca bersama, dan sel kesimpulan di
> bawah memakai OR atas ketiganya, bukan salah satu saja.

In [ ]:
def statistik(f):
    std = f.std(axis=0)
    mati = (std < 1e-5).sum()
    norma = np.linalg.norm(f, axis=1)
    c = np.cov(f, rowvar=False)
    lam = np.linalg.eigvalsh(c).clip(min=0)
    pr = (lam.sum() ** 2) / (np.square(lam).sum() + 1e-30) if lam.sum() > 0 else 0.0
    # kemiripan antar dokumen: kalau kolaps, semua dokumen terlihat sama
    rng = np.random.default_rng(0)
    a, b = rng.choice(len(f), 2000), rng.choice(len(f), 2000)
    na, nb = f[a], f[b]
    cos = (na * nb).sum(1) / (np.linalg.norm(na, axis=1) * np.linalg.norm(nb, axis=1) + 1e-12)
    return {
        "std_rata2": round(float(std.mean()), 5),
        "std_maks": round(float(std.max()), 5),
        "unit_mati": int(mati),
        "norma_rata2": round(float(norma.mean()), 4),
        "rank_efektif": round(float(pr), 2),
        "cos_antar_dokumen": round(float(np.nanmean(cos)), 4),
    }

stat = pd.DataFrame({k: statistik(f) for k, (f, _) in F.items()}).T
display(stat)

acuan = stat.loc["CNN1D standalone"]
print(f"Acuan CNN1D standalone: std {acuan['std_rata2']}, rank efektif "
      f"{acuan['rank_efektif']}, cos antar dokumen {acuan['cos_antar_dokumen']}")
print("\nCabang teks yang kolaps akan terlihat sebagai: std jauh lebih kecil, unit mati "
      "jauh lebih banyak, rank efektif mendekati 1, dan cos antar dokumen mendekati 1.")

In [ ]:
# Hipotesis C: apakah skala kedua cabang seimbang di dalam model yang sama?
print("Rasio norma citra : teks, di dalam model yang sama\n")
for m in ["concat", "noES", "sum"]:
    nt = stat.loc[f"{m}: cabang teks", "norma_rata2"]
    nc = stat.loc[f"{m}: cabang citra", "norma_rata2"]
    print(f"{m:8s} teks {nt:8.4f} | citra {nc:8.4f} | rasio citra/teks "
          f"{nc / max(nt, 1e-9):8.2f}x")
print("\nRasio jauh dari 1 berarti head didominasi cabang bernorma lebih besar tanpa "
      "pernah 'memutuskan' apa pun — hipotesis C.")

## 4. Linear probe — pengukuran yang menentukan

Classifier linear di atas fitur yang **dibekukan**. Fit pada fitur split **train**,
evaluasi pada **test**, jadi tidak ada kebocoran.

Kalau probe atas fitur teks di dalam fusion mencapai akurasi mendekati CNN1D standalone,
fiturnya **baik** dan yang salah adalah head fusion → hipotesis B. Kalau probe gagal,
cabangnya sendiri yang kolaps → hipotesis A.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

def probe(model, jenis, nama):
    ftr, ytr = ambil_fitur(model, split["train"], jenis)
    fte, yte = ambil_fitur(model, split["test"], jenis)
    sc = StandardScaler().fit(ftr)
    # tanpa multi_class: argumen itu dihapus di sklearn 1.9 dan defaultnya
    # sudah multinomial untuk solver lbfgs
    clf = LogisticRegression(max_iter=2000)
    clf.fit(sc.transform(ftr), ytr)
    pred = clf.predict(sc.transform(fte))
    return {"sumber": nama, "oa_probe": round(float((pred == yte).mean()), 4),
            "macro_f1_probe": round(float(f1_score(yte, pred, average="macro")), 4)}

baris = [
    probe(m_std, "standalone", "CNN1D standalone (acuan)"),
    probe(m_concat, "teks", "concat: cabang teks"),
    probe(m_noES, "teks", "noES: cabang teks"),
    probe(m_sum, "teks", "sum: cabang teks"),
    probe(m_concat, "citra", "concat: cabang citra"),
    probe(m_noES, "citra", "noES: cabang citra"),
]
df_probe = pd.DataFrame(baris)
display(df_probe)

acuan_probe = df_probe.iloc[0]["oa_probe"]
print(f"Acuan: probe atas fitur CNN1D standalone = {acuan_probe:.4f}")
print(f"CNN1D standalone end-to-end (dari Tahap 5) = 0.7345")
print("\nProbe seharusnya mendekati angka end-to-end itu — kalau jauh lebih rendah, "
      "ada masalah pada prosedur probe-nya sendiri, bukan pada fiturnya.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# std per dimensi, diurutkan
for k in ["CNN1D standalone", "concat: cabang teks", "noES: cabang teks",
          "sum: cabang teks"]:
    axes[0].plot(np.sort(F[k][0].std(axis=0))[::-1], label=k)
axes[0].set_yscale("log")
axes[0].set_xlabel("dimensi (diurutkan)")
axes[0].set_ylabel("std antar dokumen (log)")
axes[0].set_title("Varians per dimensi fitur teks")
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)

# norma fitur: teks vs citra per model
lbl = ["concat", "noES", "sum"]
x = np.arange(len(lbl))
axes[1].bar(x - 0.2, [stat.loc[f"{m}: cabang teks", "norma_rata2"] for m in lbl],
            0.4, label="cabang teks", color="#4C78A8")
axes[1].bar(x + 0.2, [stat.loc[f"{m}: cabang citra", "norma_rata2"] for m in lbl],
            0.4, label="cabang citra", color="#E45756")
axes[1].set_xticks(x, lbl)
axes[1].set_ylabel("norma L2 rata-rata")
axes[1].set_title("Keseimbangan skala kedua cabang")
axes[1].legend()
axes[1].grid(alpha=0.3, axis="y")

# probe
axes[2].barh(df_probe["sumber"], df_probe["oa_probe"], color="#54A24B")
axes[2].axvline(0.7345, ls="--", color="crimson", label="CNN1D end-to-end 0.7345")
axes[2].axvline(0.1782, ls=":", color="gray", label="kelas mayoritas 0.178")
axes[2].set_xlabel("OA linear probe")
axes[2].set_title("Apakah fiturnya masih membawa informasi kelas?")
axes[2].legend(fontsize=7)
axes[2].grid(alpha=0.3, axis="x")

fig.tight_layout()
save_fig(fig, "09_diagnostik_fitur_teks")

## Kesimpulan otomatis

Ambang di bawah **bukan** biner sewenang-wenang seperti di notebook 07 — ia relatif
terhadap acuan yang terukur (CNN1D standalone), dan tiga hipotesis dinilai terpisah
supaya bisa benar bersamaan.

In [ ]:
p_std = df_probe.set_index("sumber").loc["CNN1D standalone (acuan)", "oa_probe"]
p_noES = df_probe.set_index("sumber").loc["noES: cabang teks", "oa_probe"]
mayoritas = 0.1782

rasio_probe = p_noES / p_std
std_rasio = (stat.loc["noES: cabang teks", "std_rata2"]
             / max(stat.loc["CNN1D standalone", "std_rata2"], 1e-12))
rank_rasio = (stat.loc["noES: cabang teks", "rank_efektif"]
              / max(stat.loc["CNN1D standalone", "rank_efektif"], 1e-12))
skala = (stat.loc["noES: cabang citra", "norma_rata2"]
         / max(stat.loc["noES: cabang teks", "norma_rata2"], 1e-9))

print(f"probe fitur teks noES / acuan : {rasio_probe:.2f}  "
      f"({p_noES:.4f} vs {p_std:.4f})")
print(f"std fitur teks noES / acuan   : {std_rasio:.3f}")
print(f"rank efektif noES / acuan     : {rank_rasio:.3f}")
print(f"rasio skala citra/teks (noES)  : {skala:.2f}x")
print("=" * 68)

A_kolaps = std_rasio < 0.3 or rank_rasio < 0.3 or p_noES < mayoritas * 1.5
B_head = rasio_probe > 0.7 and p_noES > mayoritas * 2
C_skala = skala > 3 or skala < 1 / 3

if A_kolaps and not B_head:
    print("HIPOTESIS A didukung: cabang teks di dalam fusion KOLAPS.")
    print("Fiturnya kehilangan varians dan tidak lagi membawa informasi kelas.")
    print("\nPerbaikan yang langsung menyerang penyebabnya: inisialisasi cabang teks")
    print("dari CNN1D_seed42.pt yang sudah terlatih, alih-alih dari nol. Itu menghapus")
    print("asimetri pretrained-versus-dari-nol yang membuat cabang citra memenangkan")
    print("optimisasi sejak epoch pertama.")
elif B_head:
    print("HIPOTESIS B didukung: cabang teksnya BAIK, head fusion yang mengabaikan.")
    print(f"Probe linear atas fitur teks yang dibekukan mencapai {p_noES:.4f}, yaitu")
    print(f"{rasio_probe:.0%} dari acuan. Informasinya ada dan bisa dibaca classifier")
    print("linear — MLP fusion yang memilih tidak memakainya.")
    print("\nPerbaikan: normalisasi per cabang sebelum digabung (BatchNorm atau")
    print("LayerNorm pada masing-masing vektor 128), atau gerbang eksplisit. Bukan")
    print("inisialisasi ulang, karena cabangnya sudah baik.")
else:
    print("Tidak ada satu hipotesis pun yang didukung bersih. Angka di atas ada di")
    print("wilayah tengah; laporkan besarannya apa adanya dan jangan paksakan label.")

if C_skala:
    print(f"\nHIPOTESIS C juga didukung: skala kedua cabang tidak seimbang "
          f"({skala:.1f}x).")
    print("Ini bisa terjadi bersama A atau B, dan perbaikannya sama murahnya:")
    print("normalisasi per cabang sebelum penggabungan.")
else:
    print(f"\nHipotesis C tidak didukung: skala kedua cabang wajar ({skala:.2f}x).")
print("=" * 68)

In [ ]:
out = CONFIG["root"] / "reports" / "diagnostik_cabang_teks.csv"
gabung = stat.copy()
gabung["oa_probe"] = df_probe.set_index("sumber")["oa_probe"].reindex(gabung.index)
gabung.to_csv(out)
display(gabung)
print(f"disimpan ke {out}")

print(f"""
--- untuk disalin ke catatan ---
probe_acuan_cnn1d:: {p_std:.4f}
probe_teks_concat:: {df_probe.set_index('sumber').loc['concat: cabang teks','oa_probe']:.4f}
probe_teks_noES:: {p_noES:.4f}
probe_teks_sum:: {df_probe.set_index('sumber').loc['sum: cabang teks','oa_probe']:.4f}
probe_citra_noES:: {df_probe.set_index('sumber').loc['noES: cabang citra','oa_probe']:.4f}
std_teks_noES_per_acuan:: {std_rasio:.3f}
rank_efektif_teks_noES:: {stat.loc['noES: cabang teks','rank_efektif']}
rank_efektif_acuan:: {stat.loc['CNN1D standalone','rank_efektif']}
unit_mati_teks_noES:: {stat.loc['noES: cabang teks','unit_mati']} dari 128
rasio_skala_citra_teks:: {skala:.2f}
""")